In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("explore").getOrCreate()

df = spark.read.parquet("E:/Python/DataEngineer/MiniProject/nyc_taxi_platform/data/bronze/year=2024/month=01/yellow_tripdata_2024-01.parquet")

In [2]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [3]:
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-01 00:57:55|  2024-01-01 01:17:43|              1|         1.72|         1|                 N|         186|          79|           2|       17.7|  1.0|    0.5|       0.

In [4]:
df.filter(df.trip_distance > 10).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-01 00:49:44|  2024-01-01 01:15:47|              2|        10.82|         1|                 N|         138|         181|           1|       45.7|  6.0|    0.5|      10.

In [5]:
df.select("fare_amount", "tip_amount", "total_amount")\
    .summary("min", "max", "count")\
    .show()

+-------+-----------+----------+------------+
|summary|fare_amount|tip_amount|total_amount|
+-------+-----------+----------+------------+
|    min|     -899.0|     -80.0|      -900.0|
|    max|     5000.0|     428.0|      5000.0|
|  count|    2964624|   2964624|     2964624|
+-------+-----------+----------+------------+



In [6]:
df.select("trip_distance").summary("min", "25%", "50%", "75%", "max").show()
print(f"Số chuyến đi có khoảng cách = 0: f{df.filter(df.trip_distance <= 0).count()}")

+-------+-------------+
|summary|trip_distance|
+-------+-------------+
|    min|          0.0|
|    25%|          1.0|
|    50%|         1.68|
|    75%|         3.11|
|    max|     312722.3|
+-------+-------------+

Số chuyến đi có khoảng cách = 0: f60371


In [7]:
df.groupBy("passenger_count").count().orderBy("passenger_count").show()

+---------------+-------+
|passenger_count|  count|
+---------------+-------+
|           NULL| 140162|
|              0|  31465|
|              1|2188739|
|              2| 405103|
|              3|  91262|
|              4|  51974|
|              5|  33506|
|              6|  22353|
|              7|      8|
|              8|     51|
|              9|      1|
+---------------+-------+



In [9]:
invalid_locations = df.filter((df.PULocationID < 1) | (df.PULocationID > 265) |
                              (df.DOLocationID < 1) | (df.DOLocationID > 265))
print(f"Số chuyến đi có mã vùng không hợp lệ: {invalid_locations.count()}")

Số chuyến đi có mã vùng không hợp lệ: 0


In [10]:
from pyspark.sql.functions import col, count, when
df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
]).show()


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       0|                   0|                    0|         140162|            0|    140162|            140162|           0|           0|           0|          0|    0|      0|         